# PyTorch: MNIST Dataset Classification

In [ ]:
import torch
import random
from pathlib import Path
from torch import nn
from torch import optim
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from torchmetrics.classification import MulticlassAccuracy, MulticlassConfusionMatrix
from torchmetrics.metric import Metric
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from common import CV_DATASETS_DIR
from common.torch import (
    train, get_summary_writer,
    get_optimal_device,
    set_default_seed,
    set_default_optimizations,
    get_parameters_number
)

In [ ]:
set_default_seed()
set_default_optimizations()

In [ ]:
device = get_optimal_device();

In [ ]:
print(f"PyTorch: version {torch.__version__}")
print(f"PyTorch: {device.type.upper()} device")

In [ ]:
# Hyperparameters
BATCH_SIZE = 64
N_EPOCHS = 15

## Prepare Datasets

In [ ]:
transforms = v2.Compose([
    v2.PILToTensor(),
    v2.ToDtype(dtype=torch.float, scale=True),
])

In [ ]:
# Prepare train dataset
tr_data = datasets.FashionMNIST(
    root=CV_DATASETS_DIR/"misc",
    train=True,
    transform=transforms,
    download=True,
)

print(f"Train dataset dtype: {tr_data.data.dtype}")
print(f"Train dataset shape: {tr_data.data.shape}")

In [ ]:
# Prepare test dataset
ts_data = datasets.FashionMNIST(
    root=CV_DATASETS_DIR/"misc",
    train=False,
    transform=transforms,
    download=True
)

print(f"Test dataset dtype: {ts_data.data.dtype}")
print(f"Test dataset shape: {ts_data.data.shape}")

In [ ]:
# Prepare data loaders (data loader is a python-iterable object)
tr_dl = DataLoader(dataset=tr_data,
                   batch_size=BATCH_SIZE,
                   shuffle=False,
                   num_workers=2,
                   drop_last=True,
                   pin_memory=True)
ts_dl = DataLoader(dataset=ts_data,
                   batch_size=BATCH_SIZE,
                   shuffle=False,
                   num_workers=2,
                   drop_last=True,
                   pin_memory=True)

print(f"Train data number of batches: {len(tr_dl)}")
print(f"Test data number of batches: {len(ts_dl)}")

In [ ]:
# Display a bunch of samples
converter = v2.ToPILImage()
fig = plt.figure(figsize=(10, 10))
rows, cols = 3, 3
for i in range(1, rows * cols + 1):
    index = int(torch.randint(0, len(tr_data), size=[1]).item())
    image, label = tr_data[index]
    plt.subplot(rows, cols, i)
    plt.imshow(converter(image), cmap="gray")
    plt.title(tr_data.classes[label])
    plt.axis(False)

In [ ]:
n_classes = len(tr_data.classes)
n_classes

## Define Model

In [ ]:
class TinyVggModel(nn.Module):
    def __init__(self, in_shape: int, out_shape: int, hidden_units: int):
        super().__init__()
        self.cv_block1 = nn.Sequential(
            nn.Conv2d(in_channels=in_shape, out_channels=hidden_units, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.cv_block2 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=hidden_units*7*7,
                      out_features=out_shape),
        )

    def forward(self, inputs):
        z = self.cv_block1(inputs)
        z = self.cv_block2(z)
        outputs = self.classifier(z)
        return outputs

In [ ]:
model = TinyVggModel(in_shape=1, out_shape=n_classes, hidden_units=10).to(device)
print(f"Total model parameters: {get_parameters_number(model)}")

## Train Model

In [ ]:
logs_dir, writer = get_summary_writer("pt_mnist", "tiny_vgg", "15_epochs")
logs_dir

In [ ]:
criterion = nn.CrossEntropyLoss().to(device)
accuracy = MulticlassAccuracy(num_classes=n_classes).to(device)
# Enable fused mode for AdamW optimizer
optimizer = optim.AdamW(model.parameters(), lr=0.001, fused=True)

In [ ]:
train(model=model,
      tr_dl=tr_dl,
      ts_dl=ts_dl,
      optimizer=optimizer,
      criterion=criterion,
      metric=accuracy,
      n_epochs=N_EPOCHS,
      writer=writer,
      device=device)

## Evaluate Model

### Evaluate Test Dataset

In [ ]:
def evaluate(model: nn.Module,
             loader: DataLoader,
             criterion: nn.Module,
             metric: Metric,
             device: torch.device = device) -> dict:
    n_batches = len(loader)
    loss_avg, accu_avg = 0.0, 0.0
    model.eval()
    preds = []
    truth = []
    criterion.to(device)
    metric.to(device)
    with torch.inference_mode():
        for x, y_true in tqdm(loader):
            x, y_true = x.to(device), y_true.to(device)
            y_logits = model(x)
            y_pred = torch.softmax(y_logits, dim=1).argmax(dim=1)
            loss_avg += criterion(y_logits, y_true).item()
            metric.update(y_pred, y_true)
            preds.append(y_pred)
            truth.append(y_true)

        loss_avg = loss_avg / n_batches
        accu_avg = metric.compute().item()
    return {
        "truth": torch.flatten(torch.stack(truth)).cpu(),
        "preds": torch.flatten(torch.stack(preds)).cpu(),
        "loss": loss_avg,
        "accu": accu_avg,
    }

In [ ]:
results = evaluate(model, ts_dl, criterion, accuracy)
print(f"Loss: {results["loss"]}\nAccuracy: {results["accu"]}")

In [ ]:
metric = MulticlassConfusionMatrix(num_classes=n_classes)
metric.update(results["preds"], results["truth"])
metric.plot(labels=ts_data.classes);

### Make Predictions and Visualize

In [ ]:
def make_preds(model: nn.Module,
               samples: list,
               device: torch.device = device):
    preds = []
    model.to(device)
    model.eval()
    with torch.inference_mode():
        for sample in samples:
            sample = torch.unsqueeze(sample, dim=0).to(device)
            y_logits = model(sample)
            y_pred = torch.softmax(y_logits.squeeze(), dim=0).argmax(dim=0)
            preds.append(y_pred.item())
    return preds

In [ ]:
# Retrieve random samples
true_samples = []
true_targets = []
for sample, target in random.sample(list(zip(ts_data.data, ts_data.targets)), k=9):
    true_samples.append(torch.unsqueeze(transforms(sample), dim=0))
    true_targets.append(target.item())

In [ ]:
preds = make_preds(model, true_samples, device)
preds, true_targets

In [ ]:
plt.figure(figsize=(10, 10))
nrows = 3
ncols = 3
for i, sample in enumerate(true_samples):
    plt.subplot(nrows, ncols, i+1)
    plt.imshow(sample.squeeze(), cmap="gray")
    plt.axis(False)

    label = f"P: {tr_data.classes[preds[i]]}, T: {tr_data.classes[true_targets[i]]}"
    if preds[i] != true_targets[i]:
        plt.title(label, fontsize=10, color="r")
    else:
        plt.title(label, fontsize=10, color="g")

## Save and Load Model

In [ ]:
MODEL_PATH = Path("files")/"pt_mnist"
MODEL_PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
MODEL_NAME = 'pt_mnist.pt'
torch.save(model.state_dict(), MODEL_PATH/MODEL_NAME)

In [ ]:
model1 = TinyVggModel(in_shape=1, out_shape=n_classes, hidden_units=10)
model1.load_state_dict(torch.load(MODEL_PATH/MODEL_NAME))
model1.to(device)